In [1]:
import os
import numpy as np
import warnings 
import os
import xarray as xr
import numpy as np
import pandas as pd

from datetime import datetime

from string import Template

from experiment_configs import get_experiment_dict

ModuleNotFoundError: No module named 'experiment_configs'

In [ ]:
class WindowedTCCCalculator:
    def __init__(self, data_dir, varnames, experiment, ensemble_list, years, outdir,
                 lat_bounds=None, lon_bounds=None, landmask_file=None,
                 expected_frequency='daily'):
        self.obs_dir = reference_dir 
        self.obs_template = reference_template 
        self.data_dir = data_dir
        self.varnames = varnames
        self.experiment = experiment
        self.ensemble_list = ensemble_list
        self.years = years
        self.outdir = os.path.join(outdir, experiment)
        self.lat_bounds = lat_bounds
        self.lon_bounds = lon_bounds
        self.expected_frequency = expected_frequency
        self.windows = {
                'D01-15': slice(0, 15),
                'D16-30': slice(15, 30),
                'D31-45': slice(30, 45),
                'D46-60': slice(45, 60),
            }
            
        # Latent heat of vaporization (J/kg)
        self.Lv = 2.5e6

        # Load landmask if provided
        self.landmask = xr.open_dataset(landmask_file)['landmask'] if landmask_file else None
        os.makedirs(self.outdir, exist_ok=True)

    def build_paths(self, vname, ensemble):
        return [os.path.join(self.data_dir, f"{vname}.{ensemble}.{year}.nc") for year in self.years]

    def _load_variable(self, varname, ensemble):
        original_var = varname
        possible_vars = [varname]
        if varname == "LHFLX":
            possible_vars = ["LHFLX", "QFLX"]

        alt_dir = self.data_dir.replace("daily", "6hourly")

        for v in possible_vars:
            for base_dir in [self.data_dir, alt_dir]:
                files = [os.path.join(base_dir, f"{v}.{ensemble}.{year}.nc") for year in self.years]
                if all(os.path.exists(f) for f in files):
                    if base_dir == alt_dir:
                        self.expected_frequency = '6h'
                        print(f"[INFO] {v} found in 6-hourly path: resampling will be applied.")
                    datasets = [xr.open_dataset(f)[v] for f in files]
                    ds = xr.concat(datasets, dim='time')

                    if original_var == "LHFLX" and v == "QFLX":
                        print("[INFO] Converting QFLX to LHFLX using Lv.")
                        ds = ds * self.Lv
                        ds.attrs["units"] = "W/m2"
                        ds.name = "LHFLX"

                    return ds

        raise FileNotFoundError(f"[ERROR] Could not find {possible_vars} in {self.data_dir} or {alt_dir} for {ensemble} in daily or 6-hourly paths.")

    def _load_reference(self, refname):
        original_var = varname
        possible_vars = [varname]
        if varname == "LHFLX":
            possible_vars = ["LHFLX", "QFLX"]

        alt_dir = self.data_dir.replace("daily", "6hourly")

        for v in possible_vars:
            for base_dir in [self.data_dir, alt_dir]:
                files = [os.path.join(base_dir, f"{v}.{ensemble}.{year}.nc") for year in self.years]
                if all(os.path.exists(f) for f in files):
                    if base_dir == alt_dir:
                        self.expected_frequency = '6h'
                        print(f"[INFO] {v} found in 6-hourly path: resampling will be applied.")
                    datasets = [xr.open_dataset(f)[v] for f in files]
                    ds = xr.concat(datasets, dim='time')

                    if original_var == "LHFLX" and v == "QFLX":
                        print("[INFO] Converting QFLX to LHFLX using Lv.")
                        ds = ds * self.Lv
                        ds.attrs["units"] = "W/m2"
                        ds.name = "LHFLX"

                    return ds

        raise FileNotFoundError(f"[ERROR] Could not find {possible_vars} in {self.data_dir} or {alt_dir} for {ensemble} in daily or 6-hourly paths.")

        return self._resample_if_needed(ds)
    
    def _load_dataset(self, ensemble):
        ds = xr.Dataset({
            key: self._load_variable(vname, ensemble)
            for key, vname in self.varnames.items()
        })
        return self._resample_if_needed(ds)

    def _resample_if_needed(self, ds):
        if self.expected_frequency.lower() == '6h':
            print("[INFO] Resampling 6-hourly data to daily means...")
            return ds.resample(time='1D').mean()
        return ds

    def _subset_region(self, ds, region):
        if self.lat_bounds:
            ds = ds.sel(lat=slice(*self.lat_bounds))
        if self.lon_bounds:
            ds = ds.sel(lon=slice(*self.lon_bounds))

        if region == "land" and self.landmask is not None:
            ds = ds.where(self.landmask > 0)
        elif region == "ocean" and self.landmask is not None:
            ds = ds.where(self.landmask == 0)
        return ds

    def _compute_tcc_for_window(self, model, reference, tslice) -> xr.DataArray:
        f = model.isel(time=tslice)
        o = reference.isel(time=tslice)

        f_anom = f - f.mean(dim='time')
        o_anom = o - o.mean(dim='time')

        numerator = (f_anom * o_anom).sum(dim='time')
        denominator = np.sqrt((f_anom**2).sum(dim='time') * (o_anom**2).sum(dim='time'))

        return numerator / denominator
    

    def _save_outputs(self, label, ds, ensemble, region):
        time_index = ds.time.to_index()
        year_months = sorted(set((t.year, t.month) for t in time_index))
        
        for year, month in year_months:
            month_mask = (ds_daily.time.dt.year == year) & (ds_daily.time.dt.month == month)
            data = ds.sel(time=month_mask)
            if data.time.size == 0:
                continue

            ym_str = f"{year}-{month:02d}"
            fname = f"tcc_{label}_{ensemble}_{region}_{ym_str}.nc"
            path = os.path.join(self.outdir, fname)
            data.to_netcdf(path)

        print(f"[{self.experiment}] {ensemble} - {region.upper()}: saved tcc for {label} files")

    def process(self):
        da = self._load_reference(refname)
        if region != 'global':
            da_region = self._subset_region(da, region)
        else:
            da_region = da 
            
        for ensemble in self.ensemble_list:
            print(f"→ Processing {self.experiment} | {ensemble}")
            ds = self._load_dataset(ensemble)
            if region != 'global':
                ds_region = self._subset_region(ds, region)
            else:
                ds_region = ds
            
            for label, tslice in windows.items():
                tcc = self._compute_tcc_for_window(ds,da,tslice)
                self._save_outputs(label, tcc, ensemble, region)
                

In [ ]:
if __name__ == "__main__":

    top_path = "/compyfs/zhan391/v3_dart_cda_scratch"
    out_path = "/compyfs/zhan391/v3_dart_cda_scratch/diag_dart"
    fig_path = "/qfs/people/zhan391/e3sm_dart_work/analysis/diagnostic/figures"
    landmask_file = "/compyfs/zhan391/v3_dart_cda_scratch/diag_dart/landmask_1x1.nc"
    
    ref_base = "/compyfs/zhan391/acme_init/Observations/ERA5/6hourly"
    ref_template = "ERA5_analysis_monthly_%(year)s.nc"
    
    exp_key = {
        "v3_spinup": 2011,
        "v3_hindcast": 2012
    } #DA period 
    #exp = 'v3_spinup'
    exp = 'v3_hindcast'
    exp_dict = get_experiment_dict(exp)
    years = [exp_key[exp]] 

    varnames = {
        'LHFLX' : 'LHFLX',
        'SHFLX' : 'SHFLX',
        'FSNS'  : 'FSNS',
        'FLNS'  : 'FLNS'
    }
    
    region = "global"  # No need to loop over other regions

    for var in target_vars:
        for ens in ensemble_list:
            ym_str = f"{years[0]}-01"
            model_file = os.path.join(out_path, exp_name, f"seb_daily_{ens}_{region}_{ym_str}.nc")
            ref_file = os.path.join(ref_base, f"seb_daily_reference_{region}_{ym_str}.nc")

        if not os.path.exists(model_file) or not os.path.exists(ref_file):
            print(f"[WARNING] Missing: {model_file} or {ref_file}")
            continue

        model_data = xr.open_dataset(model_file)[var]
        ref_data = xr.open_dataset(ref_file)[var]

        tcc_calc = WindowedTCCCalculator(
            model_data=model_data,
            reference_data=ref_data,
            landmask=None,  # not applying any mask
            outdir=os.path.join(out_path, "tcc_output"),
            region=region,
            experiment=exp_name,
            ensemble=ens
        )

        tcc_calc.compute_and_save_all_windows()
        
    for exp_name, exp_meta in exp_dict.items():
        print(f"\n[INFO] Running experiment: {exp_name}")
        path = exp_meta['path']
        template_str = exp_meta['template']
        template = Template(template_str.replace('%(', '${').replace(')s', '}'))  # convert to Python Template

        nens = exp_meta['nens']
        ensemble_list = [f"EN{n:02d}" for n in range(1, nens + 1)]
        
        tcc_calc = WindowedTCCCalculator(
            model_data=model_data,
            reference_data=ref_data,
            landmask=None,  # not applying any mask
            outdir=os.path.join(out_path, "tcc_output"),
            region=region,
            experiment=exp_name,
            ensemble=ens
        )
        
        processor = WindowedTCCCalculator(
            data_dir=path,
            varnames=varnames,
            experiment=exp_name,
            ensemble_list=ensemble_list,
            years=years,
            outdir=out_path,
            lat_bounds=(-90, 90),
            lon_bounds=(0, 360),
            landmask_file=landmask_file
        )
        
        processor.process()
                   

In [ ]:
import os
import numpy as np
import xarray as xr

class WindowedTCCCalculator:
    def __init__(self, model_data: xr.DataArray, reference_data: xr.DataArray,
                 outdir: str, landmask: xr.DataArray = None, region: str = 'global',
                 experiment: str = 'EXP', ensemble: str = 'EN01'):
        """
        Parameters
        ----------
        model_data : xr.DataArray
            Forecast data [time, lat, lon]
        reference_data : xr.DataArray
            Reference data [time, lat, lon]
        outdir : str
            Output directory to save TCC maps
        landmask : xr.DataArray, optional
            1 for land, 0 for ocean
        region : str
            'global', 'land', or 'ocean'
        experiment : str
            Experiment name
        ensemble : str
            Ensemble member ID
        """
        self.model = model_data
        self.reference = reference_data
        self.region = region
        self.landmask = landmask
        self.experiment = experiment
        self.ensemble = ensemble
        self.outdir = os.path.join(outdir, experiment)
        os.makedirs(self.outdir, exist_ok=True)

        self._apply_region_mask()

    def _apply_region_mask(self):
        if self.landmask is not None and self.region in ['land', 'ocean']:
            mask = (self.landmask > 0) if self.region == 'land' else (self.landmask == 0)
            self.model = self.model.where(mask)
            self.reference = self.reference.where(mask)

    def compute_tcc_for_window(self, tslice) -> xr.DataArray:
        f = self.model.isel(time=tslice)
        o = self.reference.isel(time=tslice)

        f_anom = f - f.mean(dim='time')
        o_anom = o - o.mean(dim='time')

        numerator = (f_anom * o_anom).sum(dim='time')
        denominator = np.sqrt((f_anom**2).sum(dim='time') * (o_anom**2).sum(dim='time'))

        return numerator / denominator

    def compute_and_save_all_windows(self, windows: dict = None):
        if windows is None:
            windows = {
                'D01-15': slice(0, 15),
                'D16-30': slice(15, 30),
                'D31-45': slice(30, 45),
                'D46-60': slice(45, 60),
            }

        for label, tslice in windows.items():
            print(f"[INFO] Computing TCC for window {label}...")
            tcc = self.compute_tcc_for_window(tslice)

            fname = f"tcc_{self.experiment}_{self.ensemble}_{self.region}_{label}.nc"
            fpath = os.path.join(self.outdir, fname)

            # Assign attributes for clarity
            tcc.name = 'TCC'
            tcc.attrs['long_name'] = 'Temporal Correlation Coefficient'
            tcc.attrs['window'] = label
            tcc.attrs['region'] = self.region

            tcc.to_netcdf(fpath)
            print(f"[SAVED] {fpath}")
